# Jobs Audit: Scheduled Today & Unscheduled

This notebook audits **all Databricks jobs** in the current workspace and classifies them into two categories:

1. **Scheduled Today** — Jobs with a cron schedule that fires today (**Friday, April 17, 2026**), plus continuous and periodic jobs that would run today.
2. **Unscheduled** — Jobs with no schedule, no trigger, and no continuous configuration.

Additionally, **trigger-based** jobs (e.g., file arrival) are identified separately rather than being lumped into the unscheduled category. Paused schedules are still listed but flagged accordingly.

In [0]:
%pip install croniter --quiet

In [0]:
import requests
import json
from datetime import datetime, timedelta
import pandas as pd
from croniter import croniter

# API auth
host = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiUrl().getOrElse(None)
token = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().getOrElse(None)

headers = {"Authorization": f"Bearer {token}"}

# Fetch ALL jobs with pagination
all_jobs = []
page_token = None

while True:
    params = {"limit": 25, "expand_tasks": "false"}
    if page_token:
        params["page_token"] = page_token

    resp = requests.get(f"{host}/api/2.1/jobs/list", headers=headers, params=params)
    resp.raise_for_status()
    data = resp.json()

    jobs = data.get("jobs", [])
    all_jobs.extend(jobs)

    page_token = data.get("next_page_token")
    if not page_token:
        break

print(f"Total jobs fetched: {len(all_jobs)}")

In [0]:
from zoneinfo import ZoneInfo

TODAY = datetime(2026, 4, 17)
TODAY_END = TODAY + timedelta(days=1)

scheduled_today = []
unscheduled = []
trigger_based = []

for job in all_jobs:
    job_id = job.get("job_id")
    job_name = job.get("settings", {}).get("name", "(unnamed)")
    settings = job.get("settings", {})

    schedule = settings.get("schedule")
    trigger = settings.get("trigger")
    continuous = settings.get("continuous")
    periodic = settings.get("periodic")

    classified = False

    # --- Cron-based schedule ---
    if schedule and schedule.get("quartz_cron_expression"):
        cron_expr = schedule["quartz_cron_expression"]
        tz_str = schedule.get("timezone_id", "UTC")
        paused = schedule.get("pause_status", "UNPAUSED") == "PAUSED"

        # Convert Quartz cron (7 fields: sec min hr dom mon dow year)
        # to standard cron (5 fields: min hr dom mon dow) for croniter
        parts = cron_expr.split()
        if len(parts) >= 6:
            # Quartz: sec min hr dom mon dow [year]
            std_min = parts[1]
            std_hr = parts[2]
            std_dom = parts[3]
            std_mon = parts[4]
            std_dow = parts[5]
            # Quartz uses ? for "no specific value"; croniter uses *
            std_dom = std_dom.replace("?", "*")
            std_dow = std_dow.replace("?", "*")
            # Quartz DOW: 1=SUN..7=SAT; standard cron: 0=SUN..6=SAT
            dow_map = {"SUN": "0", "MON": "1", "TUE": "2", "WED": "3",
                       "THU": "4", "FRI": "5", "SAT": "6"}
            for name, num in dow_map.items():
                std_dow = std_dow.replace(name, num)
            # Numeric Quartz DOW shift: 1->0, 2->1, ... 7->6
            if std_dow not in ("*", ""):
                try:
                    shifted = []
                    for tok in std_dow.split(","):
                        tok = tok.strip()
                        if tok.isdigit():
                            shifted.append(str(int(tok) - 1))
                        elif "-" in tok:
                            # range like 2-6
                            lo, hi = tok.split("-", 1)
                            if lo.strip().isdigit() and hi.strip().isdigit():
                                shifted.append(f"{int(lo)-1}-{int(hi)-1}")
                            else:
                                shifted.append(tok)
                        else:
                            shifted.append(tok)
                    std_dow = ",".join(shifted)
                except Exception:
                    pass  # leave as-is if parsing fails

            std_cron = f"{std_min} {std_hr} {std_dom} {std_mon} {std_dow}"
        else:
            std_cron = cron_expr  # fallback

        try:
            tz = ZoneInfo(tz_str)
            start = TODAY.replace(tzinfo=tz)
            end = TODAY_END.replace(tzinfo=tz)
            cron = croniter(std_cron, start - timedelta(seconds=1))
            next_run = cron.get_next(datetime)
            fires_today = start <= next_run < end
        except Exception as e:
            fires_today = False
            next_run = f"parse error: {e}"

        if fires_today:
            scheduled_today.append({
                "job_id": job_id,
                "job_name": job_name,
                "schedule_type": "Cron (Paused)" if paused else "Cron",
                "cron_expression_or_details": cron_expr,
                "next_run": str(next_run)
            })
            classified = True
        elif paused:
            # Paused but would not fire today anyway — still unscheduled effectively
            pass

    # --- Continuous ---
    if continuous and not classified:
        scheduled_today.append({
            "job_id": job_id,
            "job_name": job_name,
            "schedule_type": "Continuous",
            "cron_expression_or_details": "Always running",
            "next_run": "N/A (continuous)"
        })
        classified = True

    # --- Periodic ---
    if periodic and not classified:
        interval = periodic.get("interval", "?")
        unit = periodic.get("unit", "?")
        scheduled_today.append({
            "job_id": job_id,
            "job_name": job_name,
            "schedule_type": "Periodic",
            "cron_expression_or_details": f"Every {interval} {unit}",
            "next_run": "Runs periodically"
        })
        classified = True

    # --- Trigger-based (file arrival) ---
    if trigger and not classified:
        trigger_info = json.dumps(trigger, default=str)[:120]
        trigger_based.append({
            "job_id": job_id,
            "job_name": job_name,
            "schedule_type": "Trigger-based",
            "trigger_details": trigger_info
        })
        classified = True

    # --- Unscheduled ---
    if not classified:
        unscheduled.append({
            "job_id": job_id,
            "job_name": job_name
        })

print(f"Scheduled today:  {len(scheduled_today)}")
print(f"Trigger-based:    {len(trigger_based)}")
print(f"Unscheduled:      {len(unscheduled)}")

In [0]:
# --- Scheduled Today ---
print("=" * 70)
print("JOBS SCHEDULED TO RUN TODAY (Friday, April 17, 2026)")
print("=" * 70)
if scheduled_today:
    df_scheduled = pd.DataFrame(scheduled_today)
    df_scheduled = df_scheduled.sort_values("next_run").reset_index(drop=True)
    display(df_scheduled)
else:
    print("No jobs are scheduled to run today.")

# --- Trigger-based ---
print("\n" + "=" * 70)
print("TRIGGER-BASED JOBS (e.g., file arrival)")
print("=" * 70)
if trigger_based:
    df_trigger = pd.DataFrame(trigger_based)
    display(df_trigger)
else:
    print("No trigger-based jobs found.")

# --- Unscheduled ---
print("\n" + "=" * 70)
print("UNSCHEDULED JOBS (no schedule / trigger / continuous)")
print("=" * 70)
if unscheduled:
    df_unscheduled = pd.DataFrame(unscheduled)
    display(df_unscheduled)
else:
    print("All jobs have a schedule or trigger configured.")